Test

This is a Test

In [4]:
import pandas as pd

In [12]:
# COLAB SETUP — safe to skip if running locally
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/overflow-prediction'):
        !git clone https://github.com/miirage-exe/overflow-prediction.git
    os.chdir('/content/overflow-prediction')

## Data fetching and merging

In [62]:
# Fetch pluviometer and sewage data of a specific year from the files
def fetch_datasets(year):
  df_pluv = pd.read_excel(f'data/pluviometer/{year}.xlsx')
  df_sew = pd.read_csv(f'data/sewage/U24_{year}.csv')

  # --- Cleaning of pluviometer dataset

  # Renaming columns
  labels = df_pluv.columns[1:3] # Get second and third columns name (differs between 2022 and 2023)
  df_pluv = df_pluv.rename(columns={"FLOWBRU - PLUVIO ALL":"datetime", labels[0]:"precip_avant_port", labels[1]:"precip_flagey" })

  # Keep only usefull columns and rows
  df_pluv = df_pluv[["datetime", "precip_avant_port", "precip_flagey"]]
  df_pluv = df_pluv[1:-13] # Remove first row (labels in the excel file) and last one (totals per month)

  # Converting to datetime
  df_pluv['datetime'] = pd.to_datetime(df_pluv["datetime"], format='%d-%m-%Y %H:%M')

  # Sort the dataframe by datetime, to ensure chronological order
  df_pluv = df_pluv.sort_values(by="datetime").reset_index(drop=True) # We use .reset_index(drop=True), otherwise the old index will be added as a new column, which we do not want
  df_pluv = df_pluv.set_index("datetime") # Set datetime as index of the dataframe, meaning we will use it to reference the rows

  # --- Cleaning of sewage dataset

  # Renaming columns
  df_sew = df_sew.rename(columns={"Date":"datetime", "Value":"water_level" })

  # Converting to datetime
  df_sew['datetime'] = pd.to_datetime(df_sew["datetime"])

  # Sort the dataframe by datetime, to ensure chronological order
  df_sew = df_sew.sort_values(by="datetime").reset_index(drop=True) # We use .reset_index(drop=True), otherwise the old index will be added as a new column, which we do not want
  df_sew = df_sew.set_index("datetime") # Set datetime as index of the dataframe, meaning we will use it to reference the rows


  return df_pluv, df_sew

In [63]:
def merge_dataset(year, sample_rate):
  df_pluv, df_sew = fetch_datasets(year)
  df_pluv = df_pluv.resample(sample_rate).first() # Resample to hourly data, while keeping first measure of each hour
  df_sew = df_sew.resample(sample_rate).max() # Resample to hourly data, while keeping maximum level of each hour (worst case prediction)

  return df_pluv.merge(df_sew, on="datetime")

In [64]:
# For each year, merge pluviometer and sewage data
df = merge_dataset(2022, '60min')

df.head()

,precip_avant_port,precip_flagey,water_level
datetime,,,
2022-01-01 00:00:00,0,0,883.908
2022-01-01 01:00:00,0,0,862.111
2022-01-01 02:00:00,0,0,847.735
2022-01-01 03:00:00,0,0,835.678
2022-01-01 04:00:00,0,0,801.824


## Separation into training, validation, and testing datasets